1. Librareis

In [6]:
import os
import duckdb
import pandas as pd

from pathlib import Path
from sklearn.metrics import adjusted_rand_score

from detector.config import load_config
from detector.data.cluster import cluster_near_duplicates, shingles
from detector.data.dedupe import collapse_exact
from detector.data.load import load_raw
from detector.data.split import assign_split
from detector.data.topics import assign_topics
from detector.eval.metrics import summarise, threshold_at_fpr
from detector.eval.stress import leave_one_topic_out, truncate_words
from detector.models.tfidf import fit, score

In [7]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
cfg = load_config("configs/data.yaml")
sp = cfg["split"]
df = duckdb.connect().sql("select * from 'data/processed/essays.parquet'").df()
texts = df.text.tolist()
train, val = df[df.split == "train"], df[df.split == "val"]

min_words

In [8]:
unique, _ = collapse_exact(load_raw(cfg["raw_dir"], cfg["files"]))
unique["n_words"] = unique.text.str.split().str.len()
rows = []
for m in (20, 50, 100, 200):
    d = unique[unique.n_words < m]
    rows.append({"min_words": m, "dropped": len(d), "ai": int(d.label.sum())})
pd.DataFrame(rows)

,min_words,dropped,ai
0,20,4,4
1,50,4,4
2,100,5,5
3,200,1421,28


MinHash threshold

In [9]:
rows = []
for th in (.3, .5, .7):
    sizes = pd.Series(cluster_near_duplicates(texts, 128, 5, th)).value_counts()
    rows.append({"threshold": th, "clusters": len(sizes), "largest": sizes.max(), "texts_in_2+": int(sizes[sizes > 1].sum())})
pd.DataFrame(rows)

,threshold,clusters,largest,texts_in_2+
0,0.3,17291,695,4464
1,0.5,18857,17,2747
2,0.7,19045,9,2386


read some pairs that clustered as 0.5

In [13]:
clust_05 = cluster_near_duplicates(texts, 128, 5, .5)
sizes = pd.Series(clust_05).value_counts()
for c in sizes[sizes == 2].index[:3]:
    a, b = df[clust_05 == c].text.tolist()
    sa, sb = shingles(a, 5), shingles(b, 5)
    print(round(len(sa & sb) / len(sa | sb), 2), "|", a[:70].replace("\n", " "), "|", b[:70].replace("\n", " "))

0.76 | Hot topics in today's world are pollution and global warming. While th | Hot topics in todayas world are pollution and global warming. While th
0.82 | Limiting car usage can affect our environment in so many ways. Althoug | Limiting car usage can affect our enviroment in so many ways. Although
0.96 | Dear State Senator, I send you this letter to bring up an issue that i | Dear State Senator, I send you this letter to bring up an issue that i


what is the cluster at 0.3

In [14]:
clust_03 = cluster_near_duplicates(texts, 128, 5, 0.3)
big = pd.Series(clust_03).value_counts().index[0]
g = df[clust_03 == big]
print(len(g), round(g.label.mean(), 3), g.split.value_counts().to_dict(), g.topic.value_counts().to_dict())

695 0.773 {'train': 482, 'test': 108, 'val': 105} {4: 695}


Rebuild the split at 0.3 and re-score the baseline

In [15]:
alt = df.copy()
alt["cluster_id"] = clust_03
alt["split"] = assign_split(alt, sp["seed"], sp["n_folds"], sp["train_folds"], sp["val_folds"])
print(alt.groupby("split")["label"].agg(rows="size", ai_share="mean").round(3))
tr, va = alt[alt.split == "train"], alt[alt.split == "val"]
v, c = fit(tr.text, tr.label)
s = score(v, c, va.text)
for b in (.01, .001):
    print(b, summarise(va.label, s, threshold_at_fpr(va.label, s, b)))


        rows  ai_share
split                 
test    2947     0.137
train  14349     0.172
val     2947     0.137
0.01 {'roc_auc': 0.9999580582736395, 'pr_auc': 0.9997456880229615, 'tpr': 0.9975186104218362, 'fpr': 0.009827044025157232}
0.001 {'roc_auc': 0.9999580582736395, 'pr_auc': 0.9997456880229615, 'tpr': 0.9950372208436724, 'fpr': 0.0007861635220125787}


Split seed

In [16]:
rows = []
for seed in (1, 2, 3):
    d = df.copy()
    d["split"] = assign_split(d, seed, sp["n_folds"], sp["train_folds"], sp["val_folds"])
    tr, va = d[d.split == "train"], d[d.split == "val"]
    v, c = fit(tr.text, tr.label)
    s = score(v, c, va.text)
    rows.append({"seed": seed, **summarise(va.label, s, threshold_at_fpr(va.label, s, .01))})
pd.DataFrame(rows).round(4)

,seed,roc_auc,pr_auc,tpr,fpr
0,1,0.9999,0.9997,0.998,0.0098
1,2,1.0000,0.9998,0.998,0.0098
2,3,1.0000,0.9999,1.000,0.0098


C and FPR budgets

In [19]:
rows = []
for C in (0.1, 1, 10, 100):
    v, c = fit(train.text, train.label, C=C)
    s = score(v, c, val.text)
    row = {"C": C, "roc_auc": summarise(val.label, s)["roc_auc"]}
    for b in (0.001, 0.01, .05):
        row[f"tpr@{b}"] = summarise(val.label, s, threshold_at_fpr(val.label, s, b))["tpr"]
    rows.append(row)
pd.DataFrame(rows).round(4)

,C,roc_auc,tpr@0.001,tpr@0.01,tpr@0.05
0,0.1,0.9931,0.9268,0.9472,0.9634
1,1.0,0.9999,0.9919,0.9980,1.0000
2,10.0,1.0000,1.0000,1.0000,1.0000
3,100.0,1.0000,1.0000,1.0000,1.0000


Number of Topics

In [20]:
rows = []
for k in (5, 6, 7):
    d = df.copy()
    d["alt_topic"] = assign_topics(texts, k, 100, 0)
    r = leave_one_topic_out(d, "alt_topic")
    rows.append({"k": k, "min_auc": r.roc_auc.min(), "mean_tpr": r.tpr.mean(), "min_tpr": r.tpr.min(), "max_fpr": r.fpr.max()})
pd.DataFrame(rows).round(4)

,k,min_auc,mean_tpr,min_tpr,max_fpr
0,5,0.9702,0.7839,0.4076,0.1128
1,6,0.9731,0.8299,0.4076,0.1128
2,7,0.9773,0.7888,0.3084,0.1128


In [21]:
base = df.topic.values
for dim in (50, 200):
    print("svd_dim", dim, round(adjusted_rand_score(base, assign_topics(texts, 7, dim, 0)), 3))
print("seed 1", round(adjusted_rand_score(base, assign_topics(texts, 7, 100, 1)), 3))

svd_dim 50 0.879
svd_dim 200 0.871
seed 1 0.879


In [24]:
v, c = fit(train.text, train.label)
thr = threshold_at_fpr(val.label, score(v, c, val.text), 0.01)
rows = []
for n in (20, 30, 45, 60, 80, 100, 150, 200, 300):
    s = score(v, c, val.text.map(lambda x: truncate_words(x, n)))
    rows.append({"words": n, **summarise(val.label, s, thr)})
pd.DataFrame(rows).round(3)

,words,roc_auc,pr_auc,tpr,fpr
0,20,0.961,0.895,0.986,0.501
1,30,0.976,0.933,0.986,0.396
2,45,0.991,0.975,0.992,0.289
3,60,0.998,0.991,0.998,0.214
4,80,0.999,0.997,0.998,0.152
5,100,1.000,0.999,0.998,0.114
6,150,1.000,1.000,1.000,0.053
7,200,1.000,1.000,0.998,0.026
8,300,1.000,1.000,1.000,0.015
